# ChakraNet — Milestone 2: Stage 2 Conditioning, Cropping & Residual Diffusion Downscaler
**SIH 2026 · PS 26078 · NCMRWF / MoES**

This notebook demonstrates:
1. **Dynamic Regional Cropping & Topographic Conditioning**: Extracting 12 km coarse fields and conditioning on 5 km DEM & land-sea mask.
2. **U-Net Deterministic Mean Model**: Predicting baseline orographic mean precipitation $\mu_{5\text{km}}$.
3. **CorrDiff-style Residual Diffusion**: Learning high-frequency stochastic turbulence $x_{\text{res}} = y_{\text{true}} - \mu_{5\text{km}}$.
4. **Stochastic Ensemble Realization Sampler**: Drawing 16 realizations and collapsing to severe precipitation exceedance probabilities $P(R > \tau)$.
5. **Spectral Kinetic Energy Verification**: Radially averaged 2D FFT Power Spectral Density (PSD) demonstrating high-wavenumber energy retention compared to blurred bilinear interpolation.

In [ ]:
# Setup environment and paths
import sys
from pathlib import Path
root_dir = Path.cwd().resolve()
if root_dir.name == 'notebooks':
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from models.downscaler.crop_condition import CropConditioner
from models.downscaler.unet_mean import UNetMeanPredictor
from models.downscaler.diffusion_residual import ResidualDiffusionUNet, ResidualDiffusionPipeline
from models.downscaler.sampler import DownscalerEnsembleSampler
from models.downscaler.losses import DownscalerCompositeLoss, TailWeightedCRPSLoss, RadialPSDLoss
from models.downscaler.evaluate import run_downscaler_evaluation

### Step 1: Topographic Conditioning & Regional Cropping (Odisha / AP Coast)

In [ ]:
conditioner = CropConditioner(target_size=128)
dem, land_mask = conditioner.generate_static_topography(size=128, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), facecolor='#0a0e17')
for ax in (ax1, ax2):
    ax.set_facecolor('#111827')
    ax.tick_params(colors='#94a3b8')

im1 = ax1.imshow(dem * 1500.0, cmap='terrain', vmin=0, vmax=1500)
ax1.set_title('5 km Eastern Ghats Elevation (m)', color='#e2e8f0', fontweight='bold')
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

im2 = ax2.imshow(land_mask, cmap='bone', vmin=0, vmax=1)
ax2.set_title('Land-Sea Binary Mask', color='#e2e8f0', fontweight='bold')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

### Step 2: Deterministic Mean U-Net Architecture

In [ ]:
mean_model = UNetMeanPredictor(in_channels=3, out_channels=1, base_dim=16)
dummy_input = torch.randn(1, 3, 128, 128)
pred_mean = mean_model(dummy_input)
print(f"U-Net Mean parameters: {sum(p.numel() for p in mean_model.parameters()):,}")
print(f"Mean prediction shape: {pred_mean.shape}, Min: {pred_mean.min().item():.3f} (enforcing non-negativity)")

### Step 3: Run Full Evaluation Pipeline & Spectral PSD Analysis

In [ ]:
eval_results = run_downscaler_evaluation(output_dir='../tests/artifacts')
print("Downscaler Evaluation Summary:")
for k, v in eval_results.items():
    print(f"  {k}: {v}")

# Display generated evaluation artifacts
display(Image(filename='../tests/artifacts/milestone2_downscaler_comparison.png'))
display(Image(filename='../tests/artifacts/milestone2_psd_analysis.png'))